# Defining a physical model in `Scimba`

This tutorial presents the main steps to implement the modeling of a physical problem involving partial differential equations (PDEs) in `scimba_jax`, for two kinds of problems:

- a simple Laplacian problem as seen in the previous tutorial;
- a similar problem, but with a subdomain where the diffusion coefficient changes.

## 1/ Approximation of the solution of a parametric 2D Laplacian

We consider the equation:

$$- \mu \Delta u = 2(\mu\pi)^2 \sin(\pi x) \sin(\pi y)$$

on $(x,y) \in \Omega := [-1, 1]\times[-1, 1]$ with parameter $\mu\in \mathbb{M}=[1,2]$ subject to the Dirichlet condition:

$$ u = 0 $$

on $\partial\Omega$.

The exact solution is given by

$$ u(x,y,\mu) = \mu \sin(\pi x) \sin(\pi y). $$

The **strong form** of this problem will be implemented as a **class**,
and an **instance** of this class will be used to approximate the solution numerically with a PINN.


**Remark:** the strong form of this problem is already implemented in the class `ParametricLaplacianDirichletND`
and can be directly instantiated by users, as shown in the previous tutorial.

The rest of this section describes the implementation of this class from the ground up, for pedagogical purposes.

Let us first define domains and samplers for this problem.


In [ ]:
# # uncomment the line below if you are on Google Colab
# !pip install scimba

import jax
import jax.numpy as jnp

from scimba_jax.domains.meshless_domains.domains_2d import Square2D
from scimba_jax.nonlinear_approximation.integration.monte_carlo import (
    DomainSampler,
    TensorizedSampler,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo_parameters import (
    UniformParametricSampler,
)

domain_x = Square2D([(-1.0, 1.0), (-1.0, 1.0)], is_main_domain=True)
domain_mu = [(1.0, 2.0)]
sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)], bc=True
)

## Student version: a notebook with holes

This notebook is the **student version** of `solutions/2b_models.ipynb`: some pieces of
code have been removed and replaced by calls to `TODO("...")`.

Running a cell that still contains a `TODO(...)` raises a `NotImplementedError`
with a hint. Replace each `TODO(...)` with your own code -- the surrounding text
usually contains the answer -- then re-run the cell.

The complete, filled notebook is `solutions/2b_models.ipynb`.


In [ ]:
def TODO(hint: str = ""):
    """Placeholder for a hole to fill in: raises a clear error when executed."""
    raise NotImplementedError("hole to fill" + (f" -- {hint}" if hint else ""))

In `scimba_jax`, a problem is modeled with a dictionary of residuals;
a residual is modeled by a functional operator that applies to the variable(s) $u$
of the problem, and a right-hand side.
We first introduce how variables are manipulated in `scimba_jax`.

### 1a/ Variables and Operators

In the PINN approach, an approximation of the solution $u^*$ of a problem is sought in a so-called *approximation space*, that is, a finite-dimensional parameterized space of functions $u_\theta$ that are neural networks (NNs) with weights $\theta$. An approximation is found by tuning the weights $\theta$ to minimize the residuals:
$$\mathcal{R}_i(u_\theta) = f_i \text{ for }i=1,\ldots$$
where the $\mathcal{R}_i$ are functional operators.

Implementing a residual in `scimba_jax` means implementing such a functional operator $\mathcal{R}_i$.

For a parametric stationary problem, an element of the approximation space is a callable object like:


In [ ]:
WEIGHTS_TYPE = any

def u_theta(theta: WEIGHTS_TYPE, xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    pass

Let us implement the left-hand side of a Laplacian residual $\mathcal{R}$
such that for any $u$: $$\mathcal{R}(u): {\bf x}, \mu \mapsto - \mu\Delta u({\bf x}, \mu).$$
We emphasize that $\mathcal{R}$ is an **operator**: it maps functions to functions. It should not return the evaluation of the operator at a specific point $(\bf x, \mu)$.


In [ ]:
from collections.abc import Callable


def LaplacianResidualOperator(u: Callable) -> Callable:
    """Residual operator for the parametric Laplacian problem."""
    # Hessian of u: it is a function
    hessian = TODO("differentiate u with respect to which argument? (xy is the 2nd one: argnums=1)")
    # trace of the Hessian of u: it is also a function
    trace = lambda *args: jnp.trace(hessian(*args), axis1=-2, axis2=-1)
    # parametric Laplacian operator: it remains a function
    TODO("the residual is -mu * (Laplacian of u), and the Laplacian is the trace of the Hessian")

Since we know the analytic expression of the solution of the problem, we can
verify the soundness of our implementation by defining the variable $u^*$:


In [ ]:
def f_rhs(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return 2 * (mu[..., 0:1] * jnp.pi) ** 2 * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)


def analytic_solution(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return mu * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)


def u_star(theta: WEIGHTS_TYPE, xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the PDE, used for testing purposes."""
    return analytic_solution(xy, mu)


R_u_star = LaplacianResidualOperator(u_star)

On any point of the geometric domain, $\mathcal{R}(u^*)$ and $f$ must match:


In [ ]:
N_COLLOC = 1000

key = jax.random.PRNGKey(0)  # initialize the random generator
key, sample_dict = sampler.sample(key)
xy, mu = sample_dict["interior"]

batched_f_rhs = jax.vmap(f_rhs, in_axes=(0, 0))
batched_R_u_star = jax.vmap(R_u_star, in_axes=(None, 0, 0))

assert jnp.allclose( batched_f_rhs(xy, mu), batched_R_u_star(None, xy, mu) )

### 1.b/ Implementing a functional operator in `scimba_jax`

`scimba_jax` simplifies this implementation by providing special classes for manipulating variables: the `ParamFunction` class and its subclasses:
- `ParamScalarFunction` for scalar variables (most cases),
- `ParamFieldFunction` for field variables (for instance gradients of scalar variables),
- `ParamVecFunction` for vector variables,
- `ParamMatrixFunction` mostly used internally.

`ParamFunction` instances **are** callable objects, but offer additional features.
For instance, our functional operator for the Laplacian residual could be rewritten:


In [ ]:
from scimba_jax.nonlinear_approximation.model_class.funcparam_scalar import (
    ParamScalarFunction,
)


def LaplacianResidualOperator(u: ParamScalarFunction) -> ParamScalarFunction:
    """Residual operator for the parametric Laplacian problem, like in Scimba."""
    # the Laplacian with respect to x is implemented for ParamScalarFunction instances
    lap = u.laplacian_x()
    # scalar multiplication is overloaded for ParamScalarFunction instances and plain callables
    TODO("the residual is -mu * laplacian(u); scalar multiplication is overloaded")

In order to test it, we create a `ParamScalarFunction` instance that encapsulates `u_star`
(this is just to show how this works; in practice, users do not have to create `ParamFunction`s on their own):


In [ ]:
u_star_p = ParamScalarFunction(
    {"x": 2, "mu": 1},  # dictionary of symbols: dimensions
    u_star, # the variable
    "x_mu", # the ordering of symbols in the call
)

Then apply the operator and verify:


In [ ]:
R_u_star = LaplacianResidualOperator(u_star_p)
batched_R_u_star = jax.vmap(R_u_star, in_axes=(None, 0, 0))

assert jnp.allclose( batched_f_rhs(xy, mu), batched_R_u_star(None, xy, mu) )

The functional operator will now be encapsulated in a residual.

### 1.c/ Residuals in `scimba_jax`

There are 4 types of residuals in `scimba_jax`, implemented by 4 classes:
- `InteriorResidual` models residuals of the interior of the geometric domain,
- `BoundaryResidual` models boundary conditions,
- `InitialResidual` models initial conditions,
- `DataResidual` implements data losses.

#### 1.c.1/ Implementing a parametric Laplacian residual

Let us implement a class for the parametric Laplacian operator as a subclass of `InteriorResidual`.
Such a subclass must implement a method `construct_residual` which constructs the left-hand side
of the residual for a list of variables; here, this falls back to applying the Laplacian operator.


In [ ]:
from scimba_jax.domains.meshless_domains.base import VolumetricDomain
from scimba_jax.physical_models.abstract_residuals import InteriorResidual


class ParametricLaplacianResidual(InteriorResidual):
    def __init__(self, domain: VolumetricDomain, f_rhs: Callable):
        super().__init__(domain=domain, size=1, model_type="x_mu", f_rhs=f_rhs)

    def construct_residual(self, *vars: ParamScalarFunction) -> ParamScalarFunction:
        TODO("apply the LaplacianResidualOperator to the (single) variable")

Initializing an `InteriorResidual` requires:
- `domain`: the geometric domain, a `scimba_jax` instance of `VolumetricDomain`,
- `size`: the *size* of the operator (*i.e.* the dimension of the target space of $\mathcal{R}(u)$),
- `model_type`: a `str` describing the input arguments of the variables and their order in a call,
- `f_rhs`: the right-hand side of the residual,
- for time-dependent problems, the time domain.


In [ ]:
laplacian_residual = ParametricLaplacianResidual(domain_x, f_rhs)

R_u_star = laplacian_residual.construct_residual(u_star_p)
batched_R_u_star = jax.vmap(R_u_star, in_axes=(None, 0, 0))

assert jnp.allclose(batched_f_rhs(xy, mu), batched_R_u_star(None, xy, mu))

**Remark:** When implementing the `construct_residual` method of a residual class corresponding to a functional operator $\mathcal{R}$, it is important to keep in mind that:
- `construct_residual` takes a `ParamFunction` as input and returns a `ParamFunction`; it **does not evaluate** the residual;
- the output of `construct_residual` will be *vmapped* to apply to batches of vectors hence you do not have to worry about batches;
- `construct_residual` is applied once before the training; as a consequence it would be a poor choice to seek efficiency at the expense of readability.

#### 1.c.2/ Implementing a boundary residual (homogeneous Dirichlet boundary conditions)

As a second example, we implement a subclass of `BoundaryResidual` for the Dirichlet residual defined as:
$$\mathcal{B}(u) = f$$
where the operator $\mathcal{B}$ is identity: $\mathcal{B}(u): ({\bf x}, \mu) \mapsto u({\bf x}, \mu)$.

Because the boundaries of a geometric domain can be composite, a `BoundaryResidual` instance is attached to
a *dictionary* of instances of `SurfacicDomain`, the class modeling domains of dimension $d'<d$ embedded in $\mathbb{R}^d$:


In [ ]:
from scimba_jax.domains.meshless_domains.base import SurfacicDomain
from scimba_jax.physical_models.abstract_residuals import BoundaryResidual


class DirichletResidual(BoundaryResidual):
    def __init__(self, domain: dict[str, SurfacicDomain], f_rhs):
        super().__init__(domain=domain, size=1, model_type="x_mu", f_rhs=f_rhs)

    def construct_residual(self, *vars):
        # Here assume that variables are functions of normals
        rho = vars[0]
        TODO("the Dirichlet operator is the identity")

Notice that although the model type here is `"x_mu"`, the right-hand side, if provided, is required to depend explicitly on the normals, as in:


In [ ]:
def f_bc_rhs(xy: jnp.ndarray, n: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    x = xy[..., 0:1]
    return jnp.zeros_like(x)

and it is assumed in the method `construct_residual` that the input variables also depend on the normals:


In [ ]:
def u_star_with_normals(
    theta: WEIGHTS_TYPE, xy: jnp.ndarray, n: jnp.ndarray, mu: jnp.ndarray
) -> jnp.ndarray:
    return analytic_solution(xy, mu)


u_star_wn = ParamScalarFunction(
    {"x": 2, "n": 2, "mu": 1},  # a dictionary symbol: dimension
    u_star_with_normals,  # the variable
    "x_n_mu",  # the ordering of symbols in the call
)

**Remark:** variables with normals as argument are automatically created in instances of `BoundaryResidual`.

By default, all the boundary elements of a `VolumetricDomain` that is a main domain are grouped in a dictionary labeled `"boundary"`:


In [ ]:
boundaries_dict_of_dict = domain_x.get_all_boundaries()
# by default, this returns a dictionary with a single entry, labeled "boundary",
# which is itself a dictionary
print(boundaries_dict_of_dict)

dirichlet_residual = DirichletResidual(boundaries_dict_of_dict["boundary"], f_bc_rhs)

B_u_star = dirichlet_residual.construct_residual(u_star_wn)
batched_R_u_star = jax.vmap(B_u_star, in_axes=(None, 0, 0, 0))
batched_f_bc_rhs = jax.vmap(f_bc_rhs, in_axes=(0, 0, 0))

xy, n, mu = sample_dict["boundary"]
assert jnp.allclose( batched_f_bc_rhs(xy, n, mu), batched_R_u_star(None, xy, n, mu) )

`InteriorResidual`, `BoundaryResidual` and `InitialResidual` are all subclasses of `PhysicalResidual`.

### 1.d/ Putting everything together: the physical model

We are now ready to implement the class `ParametricLaplacianDirichletND` that models the considered problem.
It will inherit from the abstract class `AbstractPhysicalModel`.

A **physical model** in `scimba_jax` is
a dictionary of `PhysicalResidual` instances
(and possibly a dictionary of `DataResidual` instances)
attached to geometric and temporal domains.

Here comes the class `ParametricLaplacianDirichletND`
which gathers the interior and boundary residuals in the dictionary of physical residuals.


In [ ]:
from scimba_jax.physical_models.abstract_physical_model import AbstractPhysicalModel


class ParametricLaplacianDirichletND(AbstractPhysicalModel):
    """A n D laplacian equation with one parameter."""

    def __init__(
        self,
        main_domain: VolumetricDomain,
        f_rhs: Callable | None = None,
        bc: str = "weak",
        f_bc_rhs: Callable | None = None,
    ):
        super().__init__(main_domain=main_domain)
        self.physical_residuals = {
            self.main_domain.get_label(): ParametricLaplacianResidual(
                domain=main_domain, f_rhs=f_rhs
            ),
        }

        if bc == "weak":
            for boundary in self.boundaries:
                self.physical_residuals[boundary] = DirichletResidual(
                    domain=self.boundaries[boundary], f_rhs=f_bc_rhs
                )

The boundary residual is created for all dictionaries of boundary elements.

Let us finally instantiate our model:


In [ ]:
model = ParametricLaplacianDirichletND(domain_x, f_rhs, bc="weak")

### 1.e/ Define and train a PINN for our physical model:

To conclude, we define and train a PINN to approximate the solution of the considered problem:


In [ ]:
import matplotlib.pyplot as plt

from scimba_jax.nonlinear_approximation.networks.mlp import MLP
from scimba_jax.nonlinear_approximation.approximation_spaces.approximation_spaces import (
    ApproximationSpace,
)
from scimba_jax.nonlinear_approximation.numerical_solvers.projectors import Projector
from scimba_jax.plots.plots_nd import plot_abstract_approx_space

key = jax.random.PRNGKey(0)  # initialize the random generator

nn = MLP(in_size=3, out_size=1, hidden_sizes=[16, 16, 16], key=key)
space = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(nn, "scalar", None)],  # nn models a scalar function u
    model_type="x_mu",  # u depends on x and mu
)

The `ApproximationSpace` object automatically generates the variables of the problem.


In [ ]:
pinn = Projector(model, space, sampler, matrix_regularization=1e-5)

key, pinn = pinn.project(key, space, 100, 1000)

plot_abstract_approx_space(
    pinn.space,  # the approximation space
    domain_x,  # the spatial domain
    domain_mu,  # the parameter's domain
    loss=pinn.losses,  # for plot of the loss: the losses
    residual=pinn.model,  # for plot of the residual: the pde
    error=analytic_solution,  # for plot of the error with respect to a func: the func
    draw_contours=True,
    n_drawn_contours=20,
)
plt.show()

## 2/ Approximation of the solution of a 2D Laplacian in a heterogeneous medium

We now consider an electrostatic problem, which consists in solving a Poisson problem on a domain composed of two nested rectangles:
$\Omega := [-1, 1]\times[-1, 1]$
and
$\Omega_{i} := [-1/2, 1/2]\times[-1/4, 1/4]$.
The two rectangles
$\Omega_{o} := \Omega \setminus \Omega_{i}$ and $\Omega_{i}$
represent two different materials.

The problem is:
$$ \begin{cases}
           - \Delta u = 0 & \text{ on } \Omega_{o}\\
           - 5\Delta u  = 0 & \text{ on } \Omega_{i}
           \end{cases}.$$

with boundary conditions on $\partial \Omega_{o}$:
$$ \begin{cases}
           \frac{\partial u}{\partial y} = 0 &\text{ on } y = -1\\
           \frac{\partial u}{\partial y} = 0 &\text{ on } y =  1\\
           u           = 1 &\text{ on } x = -1\\
           u           = -1 &\text{ on } x = 1\\
           \end{cases}$$

with a discontinuity of $\frac{\partial}{\partial x}u$ and $\frac{\partial}{\partial y}u$ on $\partial \Omega_{i}$:

$$ \left\{\begin{array}{rl}
           \lim_{ (x,y) \rightarrow \partial \Omega_{i}, (x,y) \notin \Omega_{i}}\frac{\partial}{\partial y} u &=
           3.3 \lim_{ (x,y) \rightarrow \partial \Omega_{i}, (x,y) \in \Omega_{i}}\frac{\partial}{\partial y} u
           \text{ on } y = \pm 1/4\\
           \lim_{ (x,y) \rightarrow \partial \Omega_{i}, (x,y) \notin \Omega_{i}}\frac{\partial}{\partial x} u &=
           3.3 \lim_{ (x,y) \rightarrow \partial \Omega_{i}, (x,y) \in \Omega_{i}}\frac{\partial}{\partial x} u
           \text{ on } x = \pm 1/2\\
           \end{array}\right.$$

### Domains and samplers
We first define the geometric domains $\Omega_{o}$
and
$\Omega_{i}$ as *sub-domains* of a `scimba_jax` domain representing $\Omega$:


In [ ]:
bounds_x = [(-1.0, 1.0), (-1.0, 1.0)]
bounds_ix = [(-0.5, 0.5), (-0.25, 0.25)]

omega = Square2D(bounds_x, is_main_domain=True, label_str="outer")
omega_i = Square2D(bounds_ix, is_main_domain=False, label_str="inner")

omega.add_subdomain(omega_i)

The boundary elements of `omega` and `omega_i` are respectively called:


In [ ]:
_ = tuple(print(bc.get_label()) for bc in omega.full_bc_domain())

In [ ]:
_ = tuple(print(bc.get_label()) for bc in omega_i.full_bc_domain())

We group these boundary elements in a dictionary attached to `omega`:


In [ ]:
omega.set_boundaries_dict(
    {
        "bc SN": ["bc south", "bc north"],
        "bc E": ["bc east"],
        "bc W": ["bc west"],
        "bc inner_SN": ["bc inner south", "bc inner north"],
        "bc inner_WE": ["bc inner west", "bc inner east"],
    }
)

and define a sampler:


In [ ]:
sampler = TensorizedSampler([DomainSampler(omega)], bc=True, model_type="x")

Let us try the sampler:


In [ ]:
key = jax.random.PRNGKey(0)
key, dict_of_samples = sampler.sample(key, 1000)

def plot_dict_of_samples_2d(
    dict_of_samples: dict[str, tuple[jnp.ndarray, ...]],
    fig=None, index=111
):
    if fig is None:
        fig = plt.figure()
    axe=fig.add_subplot(index)
    for label in dict_of_samples:
        x, y = dict_of_samples[label][0][:,0], dict_of_samples[label][0][:,1]
        axe.scatter(x, y, s=2, label=label)
    axe.legend()
    axe.axis('equal')
    return fig

plot_dict_of_samples_2d(dict_of_samples)
plt.show()

Notice that the samples labeled "outer" are points in $\Omega_o$, that is, in $\Omega$ but not in $\Omega_i$.

To know more about `scimba_jax` domains, follow the tutorial on domains and samplers.

### Neural Network and Unknown function

The main pitfall one might face when approximating a solution of this problem with a neural network is the discontinuity of the unknown $u$ at the interface between $\Omega_{o}$ and $\Omega_{i}$.

To tackle it,
we will approximate $u$ with a function with a two-dimensional output, say
$$ {\bf u}_{\theta} = \left( u^o_{\theta}, u^i_{\theta} \right) $$
where $u^o_{\theta}$ approximates $u$ on $\Omega_o$ and
$u^i_{\theta}$ approximates $u$ on $\Omega_i$.

Let us create an approximation space:


In [ ]:
key = jax.random.PRNGKey(0)

nn = MLP(in_size=2, out_size=2, hidden_sizes=[16, 16, 16], key=key)
space = ApproximationSpace({"x": 2}, [(nn, "vec", 2)], model_type="x")

### Residuals

The interior residual becomes a system of equations:
$$ \left\{\begin{array}{rl}
           - \Delta u^o_{\theta} &= 0 \text{ on } \Omega_{o}\\
           - 5\Delta u^i_{\theta}  &= 0 \text{ on } \Omega_{i}
           \end{array}\right.$$

We define a dedicated residual class for this scaled Laplacian residual
that will be instantiated twice, once for each equation:


In [ ]:
from scimba_jax.nonlinear_approximation.model_class.funcparam_vectorial import (
    ParamVecFunction,
)


class MultiscaleLaplacian(InteriorResidual):
    def __init__(
        self,
        scale: float,
        component: int,
        domain: VolumetricDomain,
        f_rhs: Callable | None = None,
    ):
        super().__init__(domain=domain, size=1, model_type="x", f_rhs=f_rhs)
        self.scale = scale
        self.component = component

    def construct_residual(self, *vars):
        rho = vars[0]
        assert isinstance(rho, ParamVecFunction)
        # select component
        rho_c = rho.component(self.component)
        # compute laplacian
        lap_c = rho_c.laplacian_x()
        # scale by factor
        TODO("scale the Laplacian of the selected component: -scale * laplacian")

Boundary conditions rewrite:

$$ \left\{\begin{array}{rl}
           \frac{\partial}{\partial y} u^o_{\theta} = 0 & \text{ on } y = \pm 1\\
           u^o_{\theta}           = 1 & \text{ on } x = -1\\
           u^o_{\theta}           = -1 & \text{ on } x = 1\\
           \left\{\begin{array}{r}
           u^o_{\theta}           = u^i_{\theta}\\
           \frac{\partial}{\partial y} u^o_{\theta} =
           3.3 \frac{\partial}{\partial y} u^i_{\theta}\\
           \end{array}\right. & \text{ on } \{x,y \in \partial\Omega_{i} | y = \pm 1/4 \}\\
           \left\{\begin{array}{r}
           u^o_{\theta}           = u^i_{\theta}\\
           \frac{\partial}{\partial x} u^o_{\theta} =
           3.3 \frac{\partial}{\partial x} u^i_{\theta}\\
           \end{array}\right. & \text{ on } \{x,y \in \partial\Omega_{i} | x = \pm 1/2 \}\\
\end{array}\right.$$


The first equation is a Neumann condition while the second and the third are Dirichlet conditions.

We group the remaining ones in two systems of two equations.

The Neumann condition:


In [ ]:
from scimba_jax.nonlinear_approximation.model_class.funcparam_field import (
    ParamFieldFunction,
)


class MultiscaleNeumannResidual(BoundaryResidual):
    def __init__(
        self,
        component: int,
        domain: dict[str, SurfacicDomain],
        f_rhs: Callable | None = None,
    ):
        super().__init__(domain=domain, size=1, model_type="x", f_rhs=f_rhs)
        self.component = component

    def construct_residual(self, *vars):
        rho = vars[0]
        assert isinstance(rho, ParamVecFunction)
        rho_index = rho.component(self.component)
        # grad_x_rho is a ParamFieldFunction
        grad_x_rho = rho_index.gradient_x()
        # n_func is a ParamFieldFunction taking the value of the normals
        n_func = ParamFieldFunction(
            rho.dims, lambda *args: args[rho.argnums["n"]], rho.f_type
        )
        # dot product of the gradient with the normal vector
        TODO("the normal derivative is the dot product of the gradient with the normal")

The Dirichlet condition:


In [ ]:
class MultiscaleDirichletResidual(BoundaryResidual):
    def __init__(
        self,
        component: int,
        domain: dict[str, SurfacicDomain],
        f_rhs: Callable | None = None,
    ):
        super().__init__(
            domain=domain,
            size=1,
            model_type="x",
            f_rhs=f_rhs,
            time_domain=None,
        )
        self.component = component

    def construct_residual(self, *vars):
        rho = vars[0]
        assert isinstance(rho, ParamVecFunction)
        rho_index = rho.component(self.component)
        TODO("the Dirichlet operator is the identity on the selected component")

And the interface condition:


In [ ]:
class MultiscaleInterfaceResidual(BoundaryResidual):
    def __init__(
        self,
        factor: float,
        component: int,
        domain: dict[str, SurfacicDomain],
        f_rhs: Callable | None = None,
    ):
        super().__init__(domain=domain, size=2, model_type="x", f_rhs=f_rhs)

        self.factor = factor
        self.component = component

    def construct_residual(self, *vars):
        rho = vars[0]
        assert isinstance(rho, ParamVecFunction)
        # split outputs
        rho_i = rho.component(0)
        rho_o = rho.component(1)
        # compute partial derivative with respect to symbol
        rho_i_index = rho_i.partial_derivative_x(component_index=self.component)
        rho_o_index = rho_o.partial_derivative_x(component_index=self.component)
        # scale by factor
        equality = rho_i - rho_o
        discontinuity = TODO("jump of the normal derivative across the interface: derivative on one side minus factor times derivative on the other side")
        return ParamVecFunction.cat([equality, discontinuity])

### Physical model

We group right-hand sides of residuals in dictionaries:


In [ ]:
f_rhs = {
    "inner": lambda *args: 0.0,
    "outer": lambda *args: 0.0,
}
f_bc_rhs = {
    "bc SN": lambda *args: 0.0,
    "bc E":
    TODO("u = -1 on x = 1 (east boundary)"),
    "bc W":
    TODO("u = 1 on x = -1 (west boundary)"),
    "bc inner_SN": lambda *args: 0.0,
    "bc inner_WE": lambda *args: 0.0,
}

And define the physical model corresponding to our bi-material electrostatic problem:


In [ ]:
class BiMaterialElectrostatic(AbstractPhysicalModel):
    """BiMaterialElectrostatic model."""

    def __init__(self, main_domain, f_rhs, f_bc_rhs):
        super().__init__(main_domain=main_domain)

        self.e_r = TODO("ratio of the two permittivities, given as 3.3 in the problem statement")
        # inner residuals
        self.physical_residuals = {
            "outer": MultiscaleLaplacian(
                scale=1.0, component=0, domain=main_domain, f_rhs=f_rhs["outer"]
            ),
            "inner": MultiscaleLaplacian(
                scale=TODO("diffusion coefficient in the inner rectangle (see -5 * Delta u = 0 on Omega_i)"),
                component=1, domain=main_domain, f_rhs=f_rhs["inner"]
            ),
        }
        # boundary residuals
        self.physical_residuals["bc SN"] = MultiscaleNeumannResidual(
            component=0,
            domain=self.boundaries["bc SN"],
            f_rhs=f_bc_rhs["bc SN"],
        )
        self.physical_residuals["bc E"] = MultiscaleDirichletResidual(
            component=0,
            domain=self.boundaries["bc E"],
            f_rhs=f_bc_rhs["bc E"],
        )
        self.physical_residuals["bc W"] = MultiscaleDirichletResidual(
            component=0,
            domain=self.boundaries["bc W"],
            f_rhs=f_bc_rhs["bc W"],
        )
        self.physical_residuals["bc inner_SN"] = MultiscaleInterfaceResidual(
            factor=self.e_r,
            component=1,
            domain=self.boundaries["bc inner_SN"],
            f_rhs=f_bc_rhs["bc inner_SN"],
        )
        self.physical_residuals["bc inner_WE"] = MultiscaleInterfaceResidual(
            factor=self.e_r,
            component=0,
            domain=self.boundaries["bc inner_WE"],
            f_rhs=f_bc_rhs["bc inner_WE"],
        )

Let us instantiate it:


In [ ]:
model = BiMaterialElectrostatic(omega, f_rhs, f_bc_rhs)

### Define and train a PINN for our physical model:

Finally, we define and train a PINN to approximate the solution of the problem.


In [ ]:
N_EPOCHS = 400
N_COLLOC = 2000
N_BC_COLLOC = 6000

pinn = Projector(model, space, sampler, matrix_regularization=5e-4)

key, pinn = pinn.project(
    key, space, N_EPOCHS, N_COLLOC, N_BC_COLLOC
)

plot_abstract_approx_space(
    pinn.space,
    omega,
    domain_mu,
    loss=pinn.losses,
    residual=pinn.model,
    derivatives=["ux"],
    components=[{"outer": 0, "inner": 1}], # to assemble components in a single plot
    draw_contours=True,
    n_drawn_contours=20,
    title="Bimaterial electrostatic problem",
    loss_groups=["bc"], # to group losses corresponding to boundary conditions
)
plt.show()